# DLGenAI Project 26T3 - Milestone 4
## Optimization Dynamics, Multi-Level Class Imbalance & Metric Governance

### Instructions & Setup
- **W&B Project**: `24F2004781-t32026`
- **W&B Entity**: `dineshwar-developer-indian-institute-of-technology`
- **Subset**: 1,200 unique ASCII-sorted image IDs (4,800 rows)
- **Split**: 80/20 Stratified on defect combination string (rare < 3 -> `rare_other`) -> 960 train / 240 val images
- **Canvas**: Native 128x800 canvas (Fortran order, 1-based, run clipping at 102,400)
- **Fixed Training Budget**: native 128x800 input, AdamW (lr 3e-4), batch size 16, AMP on, seed 42, decision threshold 0.5, no minimum-area filtering.

In [ ]:
# Common Environment & Path Setup
import os
import sys
import time
import gc
import ctypes
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# Apply glibc memory optimizations if running on Linux / Kaggle
try:
    ctypes.CDLL('libc.so.6').mallopt(ctypes.c_int(-100), ctypes.c_int(0))
    ctypes.CDLL('libc.so.6').mallopt(ctypes.c_int(-101), ctypes.c_int(1))
except Exception:
    pass

cv2.setNumThreads(0)
cv2.ocl.setUseOpenCL(False)

# W&B Configuration
try:
    import wandb
except ImportError:
    wandb = None

class CFG:
    SEED = 42
    WANDB_PROJECT = '24F2004781-t32026'
    WANDB_ENTITY = 'dineshwar-developer-indian-institute-of-technology'
    CANVAS_HEIGHT = 128
    CANVAS_WIDTH = 800
    CANVAS_SIZE = 128 * 800  # 102,400
    DEFECT_CLASSES = ['defect_a', 'defect_b', 'defect_c', 'defect_d']
    LR = 3e-4
    BATCH_SIZE = 16
    THRESHOLD = 0.5

# Automatic Local vs. Kaggle Path Resolution
def resolve_paths():
    candidates = [
        ('/kaggle/input/competitions/industrial-surface-defect-detection/public/train.csv', '/kaggle/input/competitions/industrial-surface-defect-detection/public/train'),
        ('/kaggle/input/industrial-surface-defect-detection/public/train.csv', '/kaggle/input/industrial-surface-defect-detection/public/train'),
        ('/kaggle/input/industrial-surface-defect-detection/train.csv', '/kaggle/input/industrial-surface-defect-detection/train'),
        ('data/train.csv', 'data/train'),
        ('../data/train.csv', '../data/train'),
    ]
    for csv_p, img_p in candidates:
        if os.path.exists(csv_p):
            return csv_p, img_p
    return 'data/train.csv', 'data/train'

DATA_CSV, IMAGE_DIR = resolve_paths()
print(f'Resolved DATA_CSV: {DATA_CSV}')
print(f'Resolved IMAGE_DIR: {IMAGE_DIR}')

# Auto-install segmentation_models_pytorch if not present (e.g. on Kaggle / Colab)
try:
    import segmentation_models_pytorch as smp
except ImportError:
    import subprocess
    print("[Setup] Installing segmentation-models-pytorch...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "segmentation-models-pytorch"])
    import segmentation_models_pytorch as smp

os.makedirs('models', exist_ok=True)


In [ ]:
# Common Setup: Working Subset, Native RLE Decoding & Stratified Split
def rle_to_mask(rle_string, height=128, width=800):
    """Decodes RLE mask on native 128x800 canvas (Fortran order, 1-based, run clipping at 102,400)."""
    if pd.isna(rle_string) or not str(rle_string).strip():
        return np.zeros((height, width), dtype=np.uint8)
    canvas_size = height * width
    mask_flat = np.zeros(canvas_size, dtype=np.uint8)
    runs = np.fromstring(str(rle_string), dtype=np.int32, sep=' ')
    starts = runs[0::2] - 1  # Convert 1-based to 0-based
    lengths = runs[1::2]
    for s, l in zip(starts, lengths):
        e = min(s + l, canvas_size)  # Clip runs exceeding canvas
        if s < canvas_size:
            mask_flat[s:e] = 1
    return mask_flat.reshape((height, width), order='F')

# 1. Load train.csv
df_all = pd.read_csv(DATA_CSV)

# 2. Sort unique image IDs alphabetically (ASCII order) and take first 1,200
unique_1200_ids = sorted(df_all['image_id'].unique())[:1200]
df_subset = df_all[df_all['image_id'].isin(unique_1200_ids)].copy().reset_index(drop=True)
print(f'Working subset: {len(unique_1200_ids)} unique image IDs, {len(df_subset)} image-class pairs.')

# 3. Build each image combination string
def get_combo(group):
    defects = group[group['mask_rle'].notna() & (group['mask_rle'].astype(str).str.strip() != '')]['defect_type'].tolist()
    if not defects:
        return 'clean'
    return '+'.join(sorted(defects))

combos = df_subset.groupby('image_id').apply(get_combo)
combo_counts = combos.value_counts()

# Merge combinations with fewer than 3 images into rare_other
rare_combos = combo_counts[combo_counts < 3].index
strat_labels = combos.apply(lambda x: 'rare_other' if x in rare_combos else x)

# 4. 80/20 Stratified Split with random_state=42
train_image_ids, val_image_ids = train_test_split(
    list(strat_labels.index),
    test_size=0.20,
    random_state=42,
    stratify=strat_labels.tolist()
)
train_image_ids = set(train_image_ids)
val_image_ids = set(val_image_ids)
print(f'Split complete: {len(train_image_ids)} train images / {len(val_image_ids)} validation images.')


---
## Question 1: Image-Level and Class-Level Imbalance

Use the 1,200-image subset (the full subset, not the split).

### Question:
(a) How many images are completely clean?
(b) How many images are positive for each of `defect_a`, `defect_b`, `defect_c`, `defect_d`?
(c) How many unique combination strings exist (before any merging)?
(d) What are the sizes of the largest and the smallest combination group?

**Answer format:** `a; b_a, b_b, b_c, b_d; c; d_largest, d_smallest` (all integers).


In [ ]:
# ==============================================================================
# Question 1: Image-Level and Class-Level Imbalance
# ==============================================================================
# Check positive status for each row
df_subset['is_positive'] = df_subset['mask_rle'].notna() & (df_subset['mask_rle'].astype(str).str.strip() != '')

# (a) Completely clean images
pos_per_image = df_subset.groupby('image_id')['is_positive'].sum()
ans_a = int((pos_per_image == 0).sum())

# (b) Positive images for each defect type
b_counts = {}
for defect in CFG.DEFECT_CLASSES:
    cnt = df_subset[(df_subset['defect_type'] == defect) & df_subset['is_positive']]['image_id'].nunique()
    b_counts[defect] = int(cnt)
ans_b_a = b_counts['defect_a']
ans_b_b = b_counts['defect_b']
ans_b_c = b_counts['defect_c']
ans_b_d = b_counts['defect_d']

# (c) Unique combination strings (before merging)
ans_c = int(combos.nunique())

# (d) Largest and smallest combination group sizes
ans_d_largest = int(combo_counts.max())
ans_d_smallest = int(combo_counts.min())

print(f'(a) Completely clean images: {ans_a}')
print(f'(b) Positive counts per defect: a={ans_b_a}, b={ans_b_b}, c={ans_b_c}, d={ans_b_d}')
print(f'(c) Unique combinations: {ans_c}')
print(f'(d) Largest group size: {ans_d_largest}, Smallest group size: {ans_d_smallest}')

# Final Formatted Answer
q1_answer = f'{ans_a}; {ans_b_a}, {ans_b_b}, {ans_b_c}, {ans_b_d}; {ans_c}; {ans_d_largest}, {ans_d_smallest}'
print('\n' + '='*40)
print(f'Question 1 Final Answer: {q1_answer}')
print('='*40)


---
## Question 2: Pixel-Level Imbalance and pos_weight

Use the same 1,200-image subset and decode all `defect_b` masks.

### Question:
(a) What percentage of all `defect_b` mask pixels (1,200 × 102,400) is positive? Give 4 decimals.
(b) What is the mean `defect_b` area (in pixels) among `defect_b`-positive images? Give 1 decimal.
(c) What is the ratio of negative to positive pixels (a candidate `pos_weight` for BCE)? Round to the nearest integer.

**Answer format:** `a (%, 4 decimals), b (1 decimal), c (integer)`.


In [ ]:
# ==============================================================================
# Question 2: Pixel-Level Imbalance and pos_weight
# ==============================================================================
b_rows = df_subset[df_subset['defect_type'] == 'defect_b'].copy()

# Decode all defect_b masks
b_areas = []
for _, row in b_rows.iterrows():
    mask = rle_to_mask(row['mask_rle'], height=CFG.CANVAS_HEIGHT, width=CFG.CANVAS_WIDTH)
    b_areas.append(int(mask.sum()))

b_areas = np.array(b_areas)
pos_b_areas = b_areas[b_areas > 0]

total_pixels = len(b_rows) * CFG.CANVAS_SIZE  # 1,200 * 102,400 = 122,880,000
total_pos_pixels = int(b_areas.sum())
total_neg_pixels = total_pixels - total_pos_pixels

# (a) Percentage of positive pixels (4 decimals)
pct_pos = (total_pos_pixels / total_pixels) * 100
ans_q2_a = f"{pct_pos:.4f}%"
ans_q2_a_num = f"{pct_pos:.4f}"

# (b) Mean defect_b area among positive images (1 decimal)
mean_b_area = float(pos_b_areas.mean())
ans_q2_b = f"{mean_b_area:.1f}"

# (c) Ratio of negative to positive pixels (nearest integer)
ratio_neg_pos = total_neg_pixels / total_pos_pixels
ans_q2_c = int(round(ratio_neg_pos))

print(f"Total pixels: {total_pixels:,}")
print(f"Total positive defect_b pixels: {total_pos_pixels:,}")
print(f"Total negative defect_b pixels: {total_neg_pixels:,}")
print(f"Number of defect_b positive images: {len(pos_b_areas)}")
print(f"(a) Percentage positive: {ans_q2_a} ({ans_q2_a_num})")
print(f"(b) Mean positive area: {ans_q2_b} px")
print(f"(c) Ratio negative/positive (pos_weight): {ans_q2_c}")

# Final Formatted Answer
q2_answer_with_pct = f"{ans_q2_a}, {ans_q2_b}, {ans_q2_c}"
q2_answer_clean = f"{ans_q2_a_num}, {ans_q2_b}, {ans_q2_c}"
print("\n" + "="*40)
print(f"Question 2 Final Answer: {q2_answer_with_pct}")
print(f"Question 2 (clean format without %): {q2_answer_clean}")
print("="*40)


---
## Question 3: Bounded Square-Root Sampling

Use the combination groups from Question 1 (all 1,200 images, no merging). Each image in group $g$ gets the weight
$$w_g = \min\left(\sqrt{\frac{n_{\text{clean}}}{n_g}}, 5\right),$$
where $n_g$ is the size of group $g$ and $n_{\text{clean}}$ is the size of the `clean` group. Draws are made with replacement, with probability proportional to weight, and one epoch is 1,200 draws.

### Question:
(a) What fraction of draws is expected to be clean, under this sampler and under natural uniform sampling?
(b) What is the expected number of draws per epoch of `defect_b`-positive images (any group containing `defect_b`), under this sampler and under natural sampling?
(c) By what factor does the sampler enrich `defect_b`-positive images?
(d) How many combination groups hit the cap of 5?

**Answer format:** `a_sampler %, a_natural % (2 decimals); b_sampler, b_natural (2 decimals); c (2 decimals, x); d (integer)`.


In [ ]:
# ==============================================================================
# Question 3: Bounded Square-Root Sampling
# ==============================================================================
# 1. Calculate weights for all 15 combination groups
n_clean = combo_counts['clean']
group_records = []

for g, n_g in combo_counts.items():
    raw_w = np.sqrt(n_clean / n_g)
    w_g = min(raw_w, 5.0)
    capped = bool(raw_w >= 5.0)
    has_b = ('defect_b' in g)
    total_w_g = n_g * w_g
    group_records.append({
        'group': g,
        'n_g': n_g,
        'raw_w': raw_w,
        'w_g': w_g,
        'capped': capped,
        'has_b': has_b,
        'total_w_g': total_w_g
    })

df_groups = pd.DataFrame(group_records)
total_weight = df_groups['total_w_g'].sum()

# (a) Fraction clean under sampler vs natural uniform sampling (in %, 2 decimals)
w_clean_total = df_groups[df_groups['group'] == 'clean']['total_w_g'].values[0]
frac_clean_sampler = (w_clean_total / total_weight) * 100
frac_clean_natural = (n_clean / 1200.0) * 100
ans_q3_a = f"{frac_clean_sampler:.2f}%, {frac_clean_natural:.2f}%"

# (b) Expected defect_b draws per epoch (1,200 draws) under sampler vs natural (2 decimals)
b_groups = df_groups[df_groups['has_b']]
w_b_total = b_groups['total_w_g'].sum()
n_b_total = b_groups['n_g'].sum()

expected_b_sampler = 1200.0 * (w_b_total / total_weight)
expected_b_natural = 1200.0 * (n_b_total / 1200.0)  # exactly n_b_total = 35
ans_q3_b = f"{expected_b_sampler:.2f}, {expected_b_natural:.2f}"

# (c) Defect_b enrichment factor (2 decimals, x)
enrichment_factor = expected_b_sampler / expected_b_natural
ans_q3_c = f"{enrichment_factor:.2f}x"

# (d) Number of combination groups hitting the cap of 5 (integer)
num_capped = int(df_groups['capped'].sum())
ans_q3_d = num_capped

print(f"Total weight of all 1,200 images: {total_weight:.4f}")
print(f"(a) Expected clean fraction: {ans_q3_a}")
print(f"(b) Expected defect_b draws per epoch: {ans_q3_b}")
print(f"(c) Defect_b enrichment factor: {ans_q3_c}")
print(f"(d) Groups hitting cap of 5: {ans_q3_d}")

# Final Formatted Answer
q3_answer = f"{ans_q3_a}; {ans_q3_b}; {ans_q3_c}; {ans_q3_d}"
print("\n" + "="*40)
print(f"Question 3 Final Answer: {q3_answer}")
print("="*40)


---
## Question 4: Warmup + Cosine Scheduler and the Stepping-Cadence Trap

Your training split has 960 images and batch size 16, so there are **60 optimizer steps per epoch**. You train for **8 epochs (480 steps)** with base $\text{LR} = 3 \times 10^{-4}$. A `LambdaLR` uses this multiplier:
$$m(t) = \begin{cases} t / 60, & t < 60 \quad (\text{linear warmup}) \\ 0.5 \left(1 + \cos\left(\pi \frac{t - 60}{420}\right)\right), & t \ge 60. \end{cases}$$

### Question:
(a) What is the learning rate at steps 15, 270, and 400?
(b) A student writes this schedule in units of steps but calls `scheduler.step()` once per epoch. What LR is the optimizer using after epoch 8?
(c) Briefly (one sentence): what symptom would you see in the training curves with this bug?

**Answer format:** `lr_15, lr_270, lr_400 (scientific notation, 3 significant figures); lr_bug (scientific notation, 3 significant figures); c (one sentence)`.


In [ ]:
# ==============================================================================
# Question 4: Warmup + Cosine Scheduler and the Stepping-Cadence Trap
# ==============================================================================
base_lr = 3e-4

def m(t):
    if t < 60:
        return t / 60.0
    else:
        return 0.5 * (1.0 + np.cos(np.pi * (t - 60.0) / 420.0))

# (a) Learning rate at steps 15, 270, and 400
lr_15 = base_lr * m(15)
lr_270 = base_lr * m(270)
lr_400 = base_lr * m(400)

ans_q4_a = f"{lr_15:.2e}, {lr_270:.2e}, {lr_400:.2e}"

# (b) LR after epoch 8 with epoch-level stepping cadence (t = 8)
t_bug = 8
lr_bug = base_lr * m(t_bug)
ans_q4_b = f"{lr_bug:.2e}"

# (c) Symptom observed in the training curve (one sentence)
ans_q4_c = "The training loss decreases extremely slowly and underfits because the learning rate remains trapped in the linear warmup phase throughout all 8 epochs, never reaching the peak learning rate."

print(f"(a) LRs at steps 15, 270, 400: {ans_q4_a}")
print(f"(b) Buggy LR after epoch 8: {ans_q4_b}")
print(f"(c) Curve symptom: {ans_q4_c}")

# Final Formatted Answer
q4_answer = f"{ans_q4_a}; {ans_q4_b}; {ans_q4_c}"
print("\n" + "="*40)
print(f"Question 4 Final Answer: {q4_answer}")
print("="*40)


---
## Question 5: Early Stopping Trace

You monitor the official validation score (`mode = max`) with `patience = 3` and `min_delta = 0.002`. An epoch counts as an improvement only if `score > best + min_delta`. Otherwise the patience counter increases by 1 (and resets to 0 on an improvement). The best checkpoint is saved only on improvement.

| Epoch | Val score |
| :---: | :---: |
| 1 | 0.512 |
| 2 | 0.601 |
| 3 | 0.648 |
| 4 | 0.661 |
| 5 | 0.664 |
| 6 | 0.668 |
| 7 | 0.671 |
| 8 | 0.670 |
| 9 | 0.672 |
| 10 | 0.669 |
| 11 | 0.673 |
| 12 | 0.665 |

### Question:
(a) At which epoch does training stop, which epoch's checkpoint is restored, and what is its score?
(b) With `min_delta = 0`, does training stop within these 12 epochs? Which epoch holds the best checkpoint at epoch 12?

**Answer format:** `e_stop, e_restored, s_restored; yes/no, e_best`.


In [ ]:
# ==============================================================================
# Question 5: Early Stopping Trace
# ==============================================================================
history_data = [
    (1, 0.512), (2, 0.601), (3, 0.648), (4, 0.661),
    (5, 0.664), (6, 0.668), (7, 0.671), (8, 0.670),
    (9, 0.672), (10, 0.669), (11, 0.673), (12, 0.665)
]

def run_early_stopping(history, patience=3, min_delta=0.002):
    best_score = -float('inf')
    best_epoch = None
    counter = 0
    stopped_epoch = None
    
    for ep, sc in history:
        if sc > best_score + min_delta:
            best_score = sc
            best_epoch = ep
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                stopped_epoch = ep
                break
    return stopped_epoch, best_epoch, best_score

# (a) Standard run with min_delta = 0.002, patience = 3
e_stop, e_restored, s_restored = run_early_stopping(history_data, patience=3, min_delta=0.002)
ans_q5_a = f"{e_stop}, {e_restored}, {s_restored:.3f}"

# (b) Run with min_delta = 0.0, patience = 3
e_stop_b, e_best_b, s_best_b = run_early_stopping(history_data, patience=3, min_delta=0.0)
stopped_within_12 = "yes" if e_stop_b is not None else "no"
ans_q5_b = f"{stopped_within_12}, {e_best_b}"

print(f"(a) Training stops at: Epoch {e_stop}, Checkpoint restored: Epoch {e_restored} (Score: {s_restored:.3f})")
print(f"(b) Stops within 12 epochs (min_delta=0)? {stopped_within_12}. Best checkpoint at epoch 12: Epoch {e_best_b} (Score: {s_best_b:.3f})")

# Final Formatted Answer
q5_answer = f"{ans_q5_a}; {ans_q5_b}"
print("\n" + "="*40)
print(f"Question 5 Final Answer: {q5_answer}")
print("="*40)


---
## Question 6: Mixed Precision, GPU Memory and Gradient Accumulation

A U-Net has 25,000,000 parameters. Assume (decimal MB, $1\text{ MB} = 10^6\text{ bytes}$):
- Weights, gradients and the two AdamW moment buffers are stored in FP32 (4 bytes each), even under AMP.
- Activations take 150 MB per sample in FP32 and exactly half of that under FP16 autocast.
- Ignore workspace, the autocast weight cache and framework overhead.

### Part A
(a) How many MB do weights + gradients + AdamW state take?
(b) What is the total memory at batch size 16 in FP32 and with AMP?
(c) With a 3,400 MB budget, what is the largest batch size that fits in FP32 and with AMP?

### Part B
You train on the 960-image training split with a micro-batch of 8 and 4 accumulation steps. The last partial batch is not dropped (check that it does not occur here).
(d) What is the effective batch size?
(e) How many micro-batches and how many optimizer steps happen per epoch?
(f) By what factor should each micro-batch loss be divided before `backward()`?
(g) How many warmup steps does a 2-epoch warmup (counted in optimizer steps) take?

**Answer format:** `a (MB); b_fp32, b_amp (MB); c_fp32, c_amp (integers); d; e_micro-batches, e_steps; f; g (integers)`.


In [ ]:
# ==============================================================================
# Question 6: Mixed Precision, GPU Memory and Gradient Accumulation
# ==============================================================================
n_params = 25_000_000
bytes_per_fp32 = 4

# Part A
# (a) Static memory: Weights (1) + Gradients (1) + AdamW 1st & 2nd moments (2) = 4 buffers
static_bytes = n_params * bytes_per_fp32 * 4
ans_a = int(static_bytes / 1e6)  # 400 MB

# (b) Total memory at batch size 16 (FP32: 150 MB/sample, AMP: 75 MB/sample)
act_fp32_per_sample = 150
act_amp_per_sample = 75
b_size = 16

ans_b_fp32 = ans_a + (b_size * act_fp32_per_sample)
ans_b_amp = ans_a + (b_size * act_amp_per_sample)

# (c) Largest batch size fitting within 3,400 MB budget
budget = 3400
act_budget = budget - ans_a  # 3000 MB available for activations
ans_c_fp32 = int(act_budget // act_fp32_per_sample)  # 3000 // 150 = 20
ans_c_amp = int(act_budget // act_amp_per_sample)    # 3000 // 75 = 40

# Part B
n_train = 960
micro_batch = 8
acc_steps = 4

# (d) Effective batch size
ans_d = micro_batch * acc_steps  # 32

# (e) Micro-batches and optimizer steps per epoch
ans_e_micro = n_train // micro_batch  # 120
ans_e_steps = ans_e_micro // acc_steps  # 30

# (f) Factor to divide loss by before backward()
ans_f = acc_steps  # 4

# (g) Warmup steps for a 2-epoch warmup (counted in optimizer steps)
ans_g = 2 * ans_e_steps  # 60

print(f"(a) Static Memory: {ans_a} MB")
print(f"(b) Total Memory at B=16: FP32={ans_b_fp32} MB, AMP={ans_b_amp} MB")
print(f"(c) Max Batch at 3,400 MB: FP32={ans_c_fp32}, AMP={ans_c_amp}")
print(f"(d) Effective batch size: {ans_d}")
print(f"(e) Per epoch: {ans_e_micro} micro-batches, {ans_e_steps} optimizer steps")
print(f"(f) Loss division factor: {ans_f}")
print(f"(g) 2-epoch warmup steps: {ans_g}")

# Final Formatted Answer
q6_answer = f"{ans_a}; {ans_b_fp32}, {ans_b_amp}; {ans_c_fp32}, {ans_c_amp}; {ans_d}; {ans_e_micro}, {ans_e_steps}; {ans_f}; {ans_g}"
print("\n" + "="*40)
print(f"Question 6 Final Answer: {q6_answer}")
print("="*40)


---
## Question 7: The Naive Dice Trap and the Official Metric

Use the empty-pair convention: $\text{Dice}(\emptyset, \emptyset) = 1$ and $\text{Dice}(\emptyset, \text{non-empty}) = 0$. The official score is
$$0.70 \cdot \text{ForegroundDice} + 0.20 \cdot \text{FalsePositiveScore} + 0.10 \cdot \text{EmptyAccuracy},$$
where $\text{FalsePositiveScore} = 1 - \text{mean}(|P| / 102,400)$ over empty pairs and $\text{EmptyAccuracy}$ is the share of empty pairs predicted fully empty.

### Part A (data-driven)
Evaluate an all-empty baseline on all 4,800 image-class pairs of the 1,200-image subset.
(a) How many pairs have an empty ground truth?
(b) What is the baseline's naive empty-aware mean Dice?
(c) What is the baseline's official score?

### Part B (toy)
A validation set has 400 pairs: 60 positive ($\text{ForegroundDice} = 0.50$) and 340 empty. The model predicts a fully empty mask on 306 empty pairs and a 1,024-pixel false blob on each of the other 34.
(d) What are the model's official score and naive Dice?
(e) Which metric ranks the model below the all-empty baseline on this set, and what is the baseline's naive Dice here?

**Answer format:** `a (integer); b, c (4 decimals); d_official, d_naive (4 decimals); e (metric name, baseline naive Dice to 4 decimals)`.


In [ ]:
# ==============================================================================
# Question 7: The Naive Dice Trap and the Official Metric
# ==============================================================================
# Part A: Data-Driven Evaluation on 4,800 pairs
total_pairs_A = len(df_subset)
pos_pairs_A = int(df_subset['is_positive'].sum())
ans_a = total_pairs_A - pos_pairs_A  # 4,039 empty pairs

# (b) All-empty baseline naive Dice on Part A (empty pairs get 1, positive get 0)
ans_b = ans_a / total_pairs_A  # 4039 / 4800 = 0.8415

# (c) All-empty baseline official score (FG Dice=0.0, FP Score=1.0, Empty Acc=1.0)
ans_c = 0.70 * 0.0 + 0.20 * 1.0 + 0.10 * 1.0  # 0.3000

# Part B: Toy Validation Set (400 pairs)
n_pos_toy = 60
n_empty_toy = 340
n_total_toy = 400
fg_dice_toy = 0.50
n_pred_empty = 306
n_pred_blob = 34
blob_px = 1024
canvas_px = 102400

# Official score components
mean_fp_ratio = (n_pred_empty * 0.0 + n_pred_blob * (blob_px / canvas_px)) / n_empty_toy
fp_score_toy = 1.0 - mean_fp_ratio  # 1.0 - 0.0010 = 0.9990
empty_acc_toy = n_pred_empty / n_empty_toy  # 306 / 340 = 0.9000
ans_d_official = 0.70 * fg_dice_toy + 0.20 * fp_score_toy + 0.10 * empty_acc_toy  # 0.6398

# Model Naive Dice on toy set
total_dice_toy = (n_pos_toy * fg_dice_toy) + (n_pred_empty * 1.0) + (n_pred_blob * 0.0)
ans_d_naive = total_dice_toy / n_total_toy  # 336.0 / 400 = 0.8400

# (e) Metric that ranks the model below baseline & baseline's naive Dice
baseline_naive_dice_toy = (n_empty_toy * 1.0 + n_pos_toy * 0.0) / n_total_toy  # 340 / 400 = 0.8500
trap_metric_name = "naive Dice"
ans_e = f"{trap_metric_name}, {baseline_naive_dice_toy:.4f}"

print(f"Part A (Data-Driven):")
print(f"(a) Empty pairs: {ans_a}")
print(f"(b) Baseline naive Dice: {ans_b:.4f}")
print(f"(c) Baseline official score: {ans_c:.4f}")

print(f"\nPart B (Toy Set):")
print(f"(d) Model official score: {ans_d_official:.4f}, Model naive Dice: {ans_d_naive:.4f}")
print(f"(e) Metric ranking model below baseline: {trap_metric_name}, Baseline naive Dice: {baseline_naive_dice_toy:.4f}")

# Final Formatted Answer
q7_answer = f"{ans_a}; {ans_b:.4f}, {ans_c:.4f}; {ans_d_official:.4f}, {ans_d_naive:.4f}; {ans_e}"
print("\n" + "="*40)
print(f"Question 7 Final Answer: {q7_answer}")
print("="*40)


---
## Question 8: Training Dynamics and Checkpoint Selection

Train your baseline on the 960/240 split for **6 epochs** using the settings in Common Setup, with the warmup + cosine schedule from Question 4. Log train loss, validation loss and the official validation score at every epoch to W&B.

### Settings & Training Budget:
- **Input**: Native $128 \times 800$ line-scan grayscale images (1-channel normalized $[0, 1]$)
- **Architecture**: `smp.Unet(encoder_name='resnet34', encoder_weights='imagenet', in_channels=1, classes=4)`
- **Optimizer**: `AdamW(lr=3e-4, weight_decay=1e-4)`
- **Batch size**: 16
- **AMP**: Enabled (`torch.cuda.amp.autocast`)
- **Seed**: 42
- **Decision threshold**: 0.5 (no minimum-area filtering)
- **Scheduler**: Linear warmup (60 steps) + cosine decay (420 steps) over 6 epochs (360 total optimizer steps)
- **W&B Logging**: `24F2004781-t32026`

### Question:
(a) At which epoch is validation loss lowest?
(b) At which epoch is the official validation score highest, and what is that score (3 decimals)?
(c) What is the generalization gap (val loss - train loss) at epoch 6 (3 decimals)?
(d) Which quantity should you monitor to select a checkpoint, and which epoch's checkpoint would you keep? Name the learning-curve pattern your run shows in one phrase (e.g., still underfitting, healthy convergence, overfitting).

**Answer format:** `a (integer); b_epoch, b_score; c (3 decimals); d (short answer)`.


In [ ]:
# ==============================================================================
# Question 8: Training Dynamics and Checkpoint Selection
# ==============================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
try:
    import segmentation_models_pytorch as smp
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "segmentation-models-pytorch"])
    import segmentation_models_pytorch as smp

# 1. Dataset for Native Grayscale 128x800 input and 4-channel binary masks
class SteelStripDataset(Dataset):
    def __init__(self, image_ids, df_all, image_dir):
        self.image_ids = list(image_ids)
        self.image_dir = image_dir
        pivoted = df_all[df_all['image_id'].isin(self.image_ids)].pivot(
            index='image_id', columns='defect_type', values='mask_rle'
        ).reset_index()
        self.pivoted = pivoted.set_index('image_id').reindex(self.image_ids).reset_index()
        
    def __len__(self):
        return len(self.image_ids)
        
    def __getitem__(self, idx):
        row = self.pivoted.iloc[idx]
        img_id = str(row['image_id'])
        img_name = img_id if img_id.endswith('.jpg') else f"{img_id}.jpg"
        img_path = os.path.join(self.image_dir, img_name)
        
        # Read grayscale image (1, 128, 800)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            img = np.zeros((128, 800), dtype=np.uint8)
        if img.shape != (128, 800):
            img = cv2.resize(img, (800, 128))
        img_t = torch.from_numpy(img).float().unsqueeze(0) / 255.0
        
        # 4-channel target mask (4, 128, 800)
        mask = np.zeros((128, 800, 4), dtype=np.float32)
        for c_idx, defect in enumerate(CFG.DEFECT_CLASSES):
            rle = row.get(defect, np.nan)
            if pd.notna(rle) and str(rle).strip():
                mask[:, :, c_idx] = rle_to_mask(str(rle), 128, 800)
        mask_t = torch.from_numpy(mask).permute(2, 0, 1)
        return img_t, mask_t

# 2. Compound Loss (0.5 BCE + 0.5 Soft-Dice)
class CompoundLoss(nn.Module):
    def __init__(self, alpha=0.5, beta=0.5, smooth=1.0):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.smooth = smooth
        self.bce = nn.BCEWithLogitsLoss()
        
    def forward(self, logits, targets):
        loss_bce = self.bce(logits, targets)
        probs = torch.sigmoid(logits)
        intersection = (probs * targets).sum(dim=(-2, -1))
        union = probs.sum(dim=(-2, -1)) + targets.sum(dim=(-2, -1))
        loss_dice = 1.0 - ((2.0 * intersection + self.smooth) / (union + self.smooth)).mean()
        return self.alpha * loss_bce + self.beta * loss_dice

# 3. Metric Evaluation: Foreground Dice, False Positive Score, Empty Accuracy & Official Score
def evaluate_validation(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    fg_dices = []
    total_empty_pairs = 0
    correct_empty_pairs = 0
    sum_fp_ratio = 0.0
    
    with torch.no_grad():
        for images, masks in dataloader:
            images, masks = images.to(device), masks.to(device)
            logits = model(images)
            loss = criterion(logits, masks)
            total_loss += loss.item() * images.size(0)
            
            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).float()  # Decision threshold 0.5, no min-area filter
            
            B, C, H, W = preds.shape
            canvas_px = H * W
            preds_flat = preds.view(B * C, -1)
            masks_flat = masks.view(B * C, -1)
            
            gt_pos = masks_flat.sum(dim=1) > 0
            pred_pos = preds_flat.sum(dim=1) > 0
            
            # Foreground Dice on positive pairs
            if gt_pos.sum() > 0:
                inter = (preds_flat[gt_pos] * masks_flat[gt_pos]).sum(dim=1)
                card = preds_flat[gt_pos].sum(dim=1) + masks_flat[gt_pos].sum(dim=1)
                dice_vals = (2.0 * inter) / (card + 1e-8)
                fg_dices.extend(dice_vals.cpu().numpy().tolist())
                
            # Empty pairs evaluation
            gt_empty = ~gt_pos
            n_empty = gt_empty.sum().item()
            if n_empty > 0:
                total_empty_pairs += n_empty
                pred_empty_on_empty = (~pred_pos[gt_empty]).sum().item()
                correct_empty_pairs += pred_empty_on_empty
                fp_px = preds_flat[gt_empty].sum(dim=1)
                sum_fp_ratio += (fp_px / canvas_px).sum().item()
                
    avg_val_loss = total_loss / len(dataloader.dataset)
    fg_dice = float(np.mean(fg_dices)) if fg_dices else 0.0
    fp_score = 1.0 - (sum_fp_ratio / total_empty_pairs) if total_empty_pairs > 0 else 1.0
    empty_acc = (correct_empty_pairs / total_empty_pairs) if total_empty_pairs > 0 else 1.0
    official_score = 0.70 * fg_dice + 0.20 * fp_score + 0.10 * empty_acc
    
    return avg_val_loss, fg_dice, fp_score, empty_acc, official_score

# 4. Training Loop for 6 Epochs
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Executing on device: {device}")

train_dataset = SteelStripDataset(train_image_ids, df_all, IMAGE_DIR)
val_dataset = SteelStripDataset(val_image_ids, df_all, IMAGE_DIR)

train_loader = DataLoader(train_dataset, batch_size=CFG.BATCH_SIZE, shuffle=True, num_workers=2 if os.name != 'nt' else 0, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_dataset, batch_size=CFG.BATCH_SIZE, shuffle=False, num_workers=2 if os.name != 'nt' else 0, pin_memory=torch.cuda.is_available())

# Initialize Baseline ResNet-34 U-Net
torch.manual_seed(CFG.SEED)
np.random.seed(CFG.SEED)
model = smp.Unet(encoder_name='resnet34', encoder_weights='imagenet', in_channels=1, classes=4).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.LR, weight_decay=1e-4)
criterion = CompoundLoss(alpha=0.5, beta=0.5, smooth=1.0)

# Warmup + Cosine Scheduler from Question 4 (stepped once per batch)
def lr_lambda(step):
    if step < 60:
        return step / 60.0
    else:
        return 0.5 * (1.0 + np.cos(np.pi * (step - 60.0) / 420.0))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
scaler = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda')) if hasattr(torch.amp, 'GradScaler') else torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

# Setup W&B
run = None
if wandb is not None:
    try:
        run = wandb.init(
            project=CFG.WANDB_PROJECT,
            entity=CFG.WANDB_ENTITY,
            name='milestone-4-q8-baseline',
            config={
                'architecture': 'ResNet34-UNet',
                'epochs': 6,
                'lr': CFG.LR,
                'batch_size': CFG.BATCH_SIZE,
                'amp': True,
                'seed': CFG.SEED
            }
        )
        print("[W&B] Run successfully initialized.")
    except Exception as e:
        print(f"[W&B] Notice: {e}. Running without online logging.")

history = {'epoch': [], 'train_loss': [], 'val_loss': [], 'fg_dice': [], 'fp_score': [], 'empty_acc': [], 'official_score': []}
best_score = -1.0
best_checkpoint_epoch = 1

# Run 6 Epochs Training Loop
print("\nStarting 6-Epoch Training Run...")
for epoch in range(1, 7):
    model.train()
    running_train_loss = 0.0
    
    for images, masks in train_loader:
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad()
        
        with (torch.amp.autocast('cuda', enabled=(device.type == 'cuda')) if hasattr(torch.amp, 'autocast') else torch.cuda.amp.autocast(enabled=(device.type == 'cuda'))):
            logits = model(images)
            loss = criterion(logits, masks)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()  # Step scheduler per batch
        
        running_train_loss += loss.item() * images.size(0)
        
    avg_train_loss = running_train_loss / len(train_dataset)
    avg_val_loss, fg_dice, fp_score, empty_acc, val_score = evaluate_validation(model, val_loader, criterion, device)
    
    history['epoch'].append(epoch)
    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['fg_dice'].append(fg_dice)
    history['fp_score'].append(fp_score)
    history['empty_acc'].append(empty_acc)
    history['official_score'].append(val_score)
    
    if val_score > best_score:
        best_score = val_score
        best_checkpoint_epoch = epoch
        os.makedirs('models', exist_ok=True)
        torch.save(model.state_dict(), 'models/q8_best_baseline.pth')
        
    print(f"Epoch {epoch}/6 | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | "
          f"FG Dice: {fg_dice:.4f} | FP: {fp_score:.4f} | Empty Acc: {empty_acc:.4f} | Official Score: {val_score:.4f}")
          
    if run is not None:
        run.log({
            'epoch': epoch,
            'train_loss': avg_train_loss,
            'val_loss': avg_val_loss,
            'foreground_dice': fg_dice,
            'false_positive_score': fp_score,
            'empty_accuracy': empty_acc,
            'official_score': val_score,
            'lr': scheduler.get_last_lr()[0]
        })

if run is not None:
    run.finish()

# 5. Programmatic Metric Answers
min_val_loss_idx = int(np.argmin(history['val_loss']))
ans_a = history['epoch'][min_val_loss_idx]

max_score_idx = int(np.argmax(history['official_score']))
ans_b_epoch = history['epoch'][max_score_idx]
ans_b_score = history['official_score'][max_score_idx]

gen_gap_ep6 = history['val_loss'][-1] - history['train_loss'][-1]
ans_c = f"{gen_gap_ep6:.3f}"

# Check learning-curve pattern: if val_loss decreases along with train_loss -> healthy convergence
val_trend = "healthy convergence" if history['val_loss'][-1] <= history['val_loss'][0] else "overfitting"
ans_d = f"official validation score, epoch {ans_b_epoch} checkpoint, {val_trend}"

print("\n" + "="*40)
print("Question 8 Summary:")
print(f"(a) Lowest val loss epoch: Epoch {ans_a} ({history['val_loss'][min_val_loss_idx]:.4f})")
print(f"(b) Highest official score epoch: Epoch {ans_b_epoch} ({ans_b_score:.3f})")
print(f"(c) Generalization gap at epoch 6: {ans_c}")
print(f"(d) Checkpoint selection & curve pattern: {ans_d}")
print("="*40)

q8_answer = f"{ans_a}; {ans_b_epoch}, {ans_b_score:.3f}; {ans_c}; {ans_d}"
print(f"\nQuestion 8 Final Answer: {q8_answer}")
print("="*40)


---
## Question 9: Defect-Size Binning and Presence vs. Overlap

Use the 1,200-image subset. For every positive image-class pair (all four classes pooled), compute the pair's total defect area (pixels in its decoded mask) and assign it to a bin: **1–100 px**, **101–500 px**, **501+ px**.

### Question:
(a) How many positive pairs fall in each bin?
(b) What percentage of all positive ground-truth pixels lies in the 501+ bin?
(c) A hypothetical model has presence recall of 30% (1–100), 70% (101–500) and 95% (501+). What is the expected total number of missed positive pairs, and which bin contributes the most missed pairs?
(d) In one sentence: why can the bin with the most missed pairs differ from the bin with the most missed pixels?

**Answer format:** `a_small, a_medium, a_large (integers); b (%, 2 decimals); c_missed (2 decimals), c_bin (bin name); d (one sentence)`.


In [ ]:
# ==============================================================================
# Question 9: Defect-Size Binning and Presence vs. Overlap
# ==============================================================================
# 1. Decode all positive pairs and compute defect area in pixels
pos_pairs_df = df_subset[df_subset['is_positive']].copy()

def compute_defect_area(rle):
    mask = rle_to_mask(rle, height=CFG.CANVAS_HEIGHT, width=CFG.CANVAS_WIDTH)
    return int(mask.sum())

pos_pairs_df['defect_area'] = pos_pairs_df['mask_rle'].apply(compute_defect_area)

# (a) Bin counts: 1-100 px, 101-500 px, 501+ px
mask_small = (pos_pairs_df['defect_area'] >= 1) & (pos_pairs_df['defect_area'] <= 100)
mask_medium = (pos_pairs_df['defect_area'] >= 101) & (pos_pairs_df['defect_area'] <= 500)
mask_large = (pos_pairs_df['defect_area'] >= 501)

ans_a_small = int(mask_small.sum())
ans_a_medium = int(mask_medium.sum())
ans_a_large = int(mask_large.sum())
ans_q9_a = f"{ans_a_small}, {ans_a_medium}, {ans_a_large}"

# (b) Percentage of positive ground-truth pixels in 501+ bin (%, 2 decimals)
total_gt_pixels = int(pos_pairs_df['defect_area'].sum())
large_gt_pixels = int(pos_pairs_df[mask_large]['defect_area'].sum())
pct_large = (large_gt_pixels / total_gt_pixels) * 100
ans_q9_b = f"{pct_large:.2f}%"

# (c) Expected missed positive pairs under recalls: 30% (small), 70% (medium), 95% (large)
missed_small = ans_a_small * (1.0 - 0.30)
missed_medium = ans_a_medium * (1.0 - 0.70)
missed_large = ans_a_large * (1.0 - 0.95)
total_missed_pairs = missed_small + missed_medium + missed_large

miss_dict = {
    '1-100 px': missed_small,
    '101-500 px': missed_medium,
    '501+ px': missed_large
}
top_missed_bin = max(miss_dict, key=miss_dict.get)
ans_q9_c = f"{total_missed_pairs:.2f}, {top_missed_bin}"

# (d) Explanation of divergence between missed pairs and missed pixels (one sentence)
ans_q9_d = "Small defect bins have low pixel counts per pair despite high missed pair rates, whereas a single missed large defect contributes thousands of pixels, decoupling missed pairs from total missed pixels."

print(f"(a) Bin counts: small={ans_a_small}, medium={ans_a_medium}, large={ans_a_large}")
print(f"(b) Percentage pixels in 501+ bin: {ans_q9_b} ({large_gt_pixels:,} / {total_gt_pixels:,} px)")
print(f"(c) Missed pairs: total={total_missed_pairs:.2f} (small={missed_small:.2f}, med={missed_medium:.2f}, large={missed_large:.2f}) | top bin={top_missed_bin}")
print(f"(d) Explanation: {ans_q9_d}")

# Final Formatted Answer
q9_answer = f"{ans_q9_a}; {ans_q9_b}; {ans_q9_c}; {ans_q9_d}"
print("\n" + "="*40)
print(f"Question 9 Final Answer: {q9_answer}")
print("="*40)


---
## Question 10: Controlled Experimentation (OFAT)

Using the same split, seed and 6-epoch budget from the Common Setup, run the four experiments below and log each one in W&B.
- **R0**: Baseline (constant LR $3 \times 10^{-4}$, natural uniform sampling, AMP on, seed 42).
- **R0b**: Identical to R0 but uses **seed 7** (used only to estimate seed noise).
- **R1**: R0 plus **warmup/cosine schedule only** (Question 4 schedule).
- **R2**: R0 plus **bounded square-root sampler only** (Question 3 weights computed on the 960 training images).

Treat $|\text{score}(R0) - \text{score}(R0b)|$ as your noise band. A change counts as a real improvement only if its gain is larger than $2\times$ this noise band.

### Question:
(a) What is the best official validation score of each run: R0, R0b, R1 and R2?
(b) What is the noise band, and what is the isolated gain of R1 and of R2 over R0?
(c) Does the gain of R1 exceed $2\times$ noise? Does the gain of R2 exceed $2\times$ noise? Answer yes or no for each.
(d) For R0 and R2, what percentage of the empty validation pairs has at least one predicted pixel (the clean-image false-alarm rate)? Use your best checkpoint and a threshold of 0.5.
(e) In one sentence, explain why a single run that turns on the sampler, warmup/cosine and AMP all at once could not tell you which component helped.
(f) In one sentence, explain what the change in false-alarm rate between R0 and R2 tells you about prior shift from sampling.

**Answer format:** `a_R0, a_R0b, a_R1, a_R2 (3 decimals); b_noise, b_gain_R1, b_gain_R2 (3 decimals); c_R1 (yes/no), c_R2 (yes/no); d_R0, d_R2 (%, 1 decimal); e (one sentence); f (one sentence)`.


In [ ]:
# ==============================================================================
# Question 10: Controlled Experimentation (OFAT)
# ==============================================================================
from torch.utils.data import WeightedRandomSampler

# 1. Compute Question 3 Bounded Square-Root Sampler Weights on the 960 Training Images
train_pivoted_df = df_subset[df_subset['image_id'].isin(train_image_ids)]
train_img_combos = combos.loc[list(train_image_ids)]
train_combo_freq = train_img_combos.value_counts()
n_clean_tr = train_combo_freq['clean']

train_group_weights = {}
for grp, cnt in train_combo_freq.items():
    w = min(np.sqrt(n_clean_tr / cnt), 5.0)
    train_group_weights[grp] = w

# 2. Generalized Training Function for OFAT Experiments
def run_ofat_experiment(exp_name, seed, use_scheduler, use_sampler, epochs=6):
    print(f"\n{'='*60}\nStarting Experiment: {exp_name} (Seed={seed}, Scheduler={use_scheduler}, Sampler={use_sampler})\n{'='*60}")
    
    # Set Seeds for exact reproducibility
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        
    tr_dataset = SteelStripDataset(train_image_ids, df_all, IMAGE_DIR)
    va_dataset = SteelStripDataset(val_image_ids, df_all, IMAGE_DIR)
    
    # Sampler configuration
    if use_sampler:
        # Weight per image in dataset order
        sample_weights = [train_group_weights[train_img_combos.loc[img_id]] for img_id in tr_dataset.image_ids]
        sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)
        tr_loader = DataLoader(tr_dataset, batch_size=CFG.BATCH_SIZE, sampler=sampler, num_workers=2 if os.name != 'nt' else 0, pin_memory=torch.cuda.is_available())
    else:
        tr_loader = DataLoader(tr_dataset, batch_size=CFG.BATCH_SIZE, shuffle=True, num_workers=2 if os.name != 'nt' else 0, pin_memory=torch.cuda.is_available())
        
    va_loader = DataLoader(va_dataset, batch_size=CFG.BATCH_SIZE, shuffle=False, num_workers=2 if os.name != 'nt' else 0, pin_memory=torch.cuda.is_available())
    
    # Model & Optimizer
    exp_model = smp.Unet(encoder_name='resnet34', encoder_weights='imagenet', in_channels=1, classes=4).to(device)
    exp_opt = torch.optim.AdamW(exp_model.parameters(), lr=CFG.LR, weight_decay=1e-4)
    exp_crit = CompoundLoss(alpha=0.5, beta=0.5, smooth=1.0)
    exp_scaler = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda')) if hasattr(torch.amp, 'GradScaler') else torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))
    
    exp_sched = None
    if use_scheduler:
        exp_sched = torch.optim.lr_scheduler.LambdaLR(exp_opt, lr_lambda=lr_lambda)
        
    # W&B Logging
    exp_run = None
    if wandb is not None:
        try:
            exp_run = wandb.init(
                project=CFG.WANDB_PROJECT,
                entity=CFG.WANDB_ENTITY,
                name=f"m4-q10-{exp_name}",
                config={
                    'experiment': exp_name,
                    'seed': seed,
                    'scheduler': use_scheduler,
                    'sampler': use_sampler,
                    'epochs': epochs
                }
            )
        except Exception as e:
            print(f"[W&B Notice] {e}")
            
    best_val_score = -1.0
    best_ckpt_state = None
    best_empty_acc = 1.0
    
    for epoch in range(1, epochs + 1):
        exp_model.train()
        running_loss = 0.0
        
        for imgs, msks in tr_loader:
            imgs, msks = imgs.to(device), msks.to(device)
            exp_opt.zero_grad()
            
            with (torch.amp.autocast('cuda', enabled=(device.type == 'cuda')) if hasattr(torch.amp, 'autocast') else torch.cuda.amp.autocast(enabled=(device.type == 'cuda'))):
                out = exp_model(imgs)
                loss = exp_crit(out, msks)
                
            exp_scaler.scale(loss).backward()
            exp_scaler.step(exp_opt)
            exp_scaler.update()
            if exp_sched is not None:
                exp_sched.step()
                
            running_loss += loss.item() * imgs.size(0)
            
        tr_loss = running_loss / len(tr_dataset)
        val_loss, fg_d, fp_s, emp_a, score = evaluate_validation(exp_model, va_loader, exp_crit, device)
        
        if score > best_val_score:
            best_val_score = score
            best_empty_acc = emp_a
            best_ckpt_state = exp_model.state_dict().copy()
            
        print(f"[{exp_name}] Epoch {epoch}/{epochs} | Train Loss: {tr_loss:.4f} | Val Loss: {val_loss:.4f} | "
              f"FG Dice: {fg_d:.4f} | FP: {fp_s:.4f} | Empty Acc: {emp_a:.4f} | Official Score: {score:.4f}")
              
        if exp_run is not None:
            exp_run.log({
                'epoch': epoch,
                'train_loss': tr_loss,
                'val_loss': val_loss,
                'foreground_dice': fg_d,
                'false_positive_score': fp_s,
                'empty_accuracy': emp_a,
                'official_score': score
            })
            
    if exp_run is not None:
        exp_run.finish()
        
    # Evaluate Clean-Image False-Alarm Rate on best checkpoint
    # False alarm rate = (1.0 - best_empty_acc) * 100%
    false_alarm_rate_pct = (1.0 - best_empty_acc) * 100.0
    
    return best_val_score, false_alarm_rate_pct

# 3. Execute the 4 OFAT Experiments
score_R0, fa_R0   = run_ofat_experiment('R0',   seed=42, use_scheduler=False, use_sampler=False)
score_R0b, fa_R0b = run_ofat_experiment('R0b',  seed=7,  use_scheduler=False, use_sampler=False)
score_R1, fa_R1   = run_ofat_experiment('R1',   seed=42, use_scheduler=True,  use_sampler=False)
score_R2, fa_R2   = run_ofat_experiment('R2',   seed=42, use_scheduler=False, use_sampler=True)

# 4. Calculate OFAT Metrics & Comparisons
# (a) Best official scores
ans_q10_a = f"{score_R0:.3f}, {score_R0b:.3f}, {score_R1:.3f}, {score_R2:.3f}"

# (b) Noise band and isolated gains
noise_band = abs(score_R0 - score_R0b)
gain_R1 = score_R1 - score_R0
gain_R2 = score_R2 - score_R0
ans_q10_b = f"{noise_band:.3f}, {gain_R1:.3f}, {gain_R2:.3f}"

# (c) Exceeds 2x noise threshold?
c_R1_exceeds = "yes" if gain_R1 > (2.0 * noise_band) else "no"
c_R2_exceeds = "yes" if gain_R2 > (2.0 * noise_band) else "no"
ans_q10_c = f"{c_R1_exceeds}, {c_R2_exceeds}"

# (d) Clean-image false alarm rate for R0 and R2 (%, 1 decimal)
ans_q10_d = f"{fa_R0:.1f}%, {fa_R2:.1f}%"

# (e) Why changing all at once cannot isolate components (one sentence)
ans_q10_e = "Varying multiple components simultaneously creates confounding interactions, preventing causal attribution of metric gains to any individual technique."

# (f) What change in false alarm rate tells about prior shift (one sentence)
ans_q10_f = "Artificially upsampling defect classes shifts the model's predictive prior toward positive predictions, increasing false-positive blob alarms on clean background images."

print("\n" + "="*60)
print("OFAT Experiment Results Summary:")
print(f"(a) Best Official Scores: R0={score_R0:.3f}, R0b={score_R0b:.3f}, R1={score_R1:.3f}, R2={score_R2:.3f}")
print(f"(b) Noise Band: {noise_band:.3f} | Gain R1: {gain_R1:.3f} | Gain R2: {gain_R2:.3f}")
print(f"(c) Exceeds 2x Noise: R1={c_R1_exceeds}, R2={c_R2_exceeds}")
print(f"(d) False Alarm Rates: R0={fa_R0:.1f}%, R2={fa_R2:.1f}%")
print(f"(e) Confounding Explanation: {ans_q10_e}")
print(f"(f) Prior Shift Explanation: {ans_q10_f}")
print("="*60)

# Final Formatted Answer
q10_answer = f"{ans_q10_a}; {ans_q10_b}; {ans_q10_c}; {ans_q10_d}; {ans_q10_e}; {ans_q10_f}"
print(f"\nQuestion 10 Final Answer: {q10_answer}")
print("="*60)
